# StyleFinder — Task 2 Results Visualization

Run this notebook **after** `build_index.py` has been executed. It loads all indexes from disk and generates visualizations for the presentation.

In [ ]:
%cd /content/stylefinder
!pip install -q umap-learn plotly matplotlib

In [ ]:
import sys, os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import faiss
import sqlite3
import umap
import plotly.express as px
from sentence_transformers import SentenceTransformer
from IPython.display import display, HTML

sys.path.insert(0, ".")
from src.config import *
from src.search import StyleFinderEngine

In [ ]:
engine = StyleFinderEngine()

# Also load raw data for visualizations
from datasets import load_dataset
product_dataset = load_dataset("ashraq/fashion-product-images-small", split="train")
product_df = product_dataset.to_pandas()
product_df = product_df.dropna(subset=['productDisplayName', 'articleType']).reset_index(drop=True)
style_df = load_dataset("neuralwork/fashion-style-instruct", split="train").to_pandas()

# Load embeddings from FAISS (reconstruct)
style_faiss = faiss.read_index(STYLE_FAISS)
product_faiss = faiss.read_index(PRODUCT_FAISS)
n_style = style_faiss.ntotal
n_product = product_faiss.ntotal
dim = 384

style_embeddings = np.vstack([style_faiss.reconstruct(i) for i in range(n_style)]).astype('float32')
product_embeddings = np.vstack([product_faiss.reconstruct(i) for i in range(n_product)]).astype('float32')
print(f"Style embeddings: {style_embeddings.shape}")
print(f"Product embeddings: {product_embeddings.shape}")

## 1. Index Summary

In [ ]:
# File sizes
files = []
for fname in sorted(os.listdir(INDEX_DIR)):
    size = os.path.getsize(os.path.join(INDEX_DIR, fname)) / 1024 / 1024
    files.append({"File": fname, "Size (MB)": round(size, 2)})

df_files = pd.DataFrame(files)
total = df_files["Size (MB)"].sum()
df_files.loc[len(df_files)] = {"File": "TOTAL", "Size (MB)": round(total, 2)}

display(HTML("<h4>Persisted Index Files</h4>"))
display(df_files.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

# Summary stats
summary = {
    "Index": ["FAISS (style)", "FAISS (product)", "SQLite (products)", "SQLite (styles)"],
    "Items": [n_style, n_product, len(product_df), len(style_df)],
    "Type": ["384-dim vectors", "384-dim vectors", "Metadata rows", "Metadata rows"],
    "Search": ["Semantic", "Semantic", "SQL filter", "SQL filter"]
}
display(HTML("<h4>Index Summary</h4>"))
display(pd.DataFrame(summary).style.hide(axis='index'))

## 2. Product Catalog Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Gender distribution
gender_counts = product_df['gender'].value_counts()
axes[0].barh(gender_counts.index, gender_counts.values, color=['#1E2761', '#F96167', '#4CAF50', '#FF9800', '#9C27B0'])
axes[0].set_title("Products by Gender", fontweight='bold')
axes[0].invert_yaxis()

# Top 8 article types
top_types = product_df['articleType'].value_counts().head(8)
axes[1].barh(top_types.index, top_types.values, color='#1E2761')
axes[1].set_title("Top 8 Article Types", fontweight='bold')
axes[1].invert_yaxis()

# Usage
usage_counts = product_df['usage'].value_counts()
axes[2].pie(usage_counts.values, labels=usage_counts.index, autopct='%1.0f%%',
            colors=['#1E2761', '#F96167', '#4CAF50', '#FF9800', '#9C27B0'])
axes[2].set_title("Usage Distribution", fontweight='bold')

plt.tight_layout()
plt.savefig("product_overview.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Style Dataset Overview

In [ ]:
# Extract occasions from style data
occasions = style_df['context'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(occasions.index, occasions.values, color='#F96167')
ax.set_title("Top 12 Occasions in Style Dataset", fontweight='bold')
ax.set_xlabel("Number of entries")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("style_occasions.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. UMAP: Product Embeddings by Category

In [ ]:
np.random.seed(42)
sample_n = 1000
sample_idx = np.random.choice(n_product, sample_n, replace=False)
sample_embs = product_embeddings[sample_idx]

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
umap_2d = reducer.fit_transform(sample_embs)

categories = [product_df.iloc[i]['masterCategory'] for i in sample_idx]
cat_map = {'Apparel': '#E53935', 'Footwear': '#1E88E5', 'Accessories': '#43A047',
           'Personal Care': '#8E24AA', 'Free Items': '#FB8C00', 'Sporting Goods': '#00897B', 'Home': '#F4511E'}

fig, ax = plt.subplots(figsize=(12, 8))
for cat, color in cat_map.items():
    mask = [c == cat for c in categories]
    if any(mask):
        pts = umap_2d[mask]
        ax.scatter(pts[:, 0], pts[:, 1], c=color, label=cat, s=15, alpha=0.6)

ax.legend(fontsize=10, markerscale=3)
ax.set_title("UMAP: Product Embeddings by Master Category (n=1000)", fontsize=14, fontweight='bold')
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("umap_products.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. UMAP: Style Recommendations

In [ ]:
reducer_s = umap.UMAP(n_neighbors=10, min_dist=0.1, n_components=2, random_state=42)
style_2d = reducer_s.fit_transform(style_embeddings)

# Color by occasion type (group similar ones)
def occasion_group(ctx):
    ctx = str(ctx).lower()
    if 'wedding' in ctx: return 'Wedding'
    if 'interview' in ctx or 'business' in ctx or 'work' in ctx or 'office' in ctx: return 'Professional'
    if 'date' in ctx: return 'Date'
    if 'party' in ctx or 'club' in ctx or 'gala' in ctx: return 'Party/Night'
    if 'hiking' in ctx or 'camping' in ctx or 'national' in ctx or 'retreat' in ctx: return 'Outdoor'
    if 'beach' in ctx or 'tropical' in ctx or 'cruise' in ctx: return 'Beach/Travel'
    return 'Other'

occ_groups = [occasion_group(style_df.iloc[i]['context']) for i in range(len(style_df))]
occ_colors = {'Wedding': '#E53935', 'Professional': '#1E88E5', 'Date': '#F06292',
              'Party/Night': '#8E24AA', 'Outdoor': '#43A047', 'Beach/Travel': '#00BCD4', 'Other': '#BDBDBD'}

fig, ax = plt.subplots(figsize=(12, 8))
for occ, color in occ_colors.items():
    mask = [o == occ for o in occ_groups]
    if any(mask):
        pts = style_2d[mask]
        ax.scatter(pts[:, 0], pts[:, 1], c=color, label=occ, s=12, alpha=0.5)

ax.legend(fontsize=10, markerscale=3)
ax.set_title("UMAP: Style Recommendations by Occasion Type", fontsize=14, fontweight='bold')
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("umap_styles.png", dpi=150, bbox_inches='tight')
plt.show()

## 6. StyleFinder Demo with Product Images

In [ ]:
def stylefinder_visual(query, n_styles=2, n_products=5):
    print(f"\nQuery: {query}")
    print("=" * 70)

    # Layer 1
    style_results = engine.faiss_search(query, layer="style", n=n_styles)
    print("\n--- LAYER 1: Style Recommendations ---")
    for r in style_results:
        # Get the original style entry
        sid = int(r['doc_id'])
        row = style_df.iloc[sid]
        print(f"\n  Score: {r['score']:.4f}")
        print(f"  Profile: {row['input'][:100]}...")
        print(f"  Occasion: {row['context']}")
        # Show first outfit only
        comp = row['completion']
        end = comp.find("Outfit 2")
        if end == -1: end = comp.find("2. Outfit")
        if end == -1: end = comp.find("Outfit Combination 2")
        if end == -1: end = 400
        print(f"  Outfit: {comp[:end].strip()[:300]}")

    # Layer 2
    best_text = engine.style_corpus[style_results[0]["doc_id"]][:300]
    product_query = query + " " + best_text
    product_results = engine.faiss_search(product_query, layer="product", n=n_products)

    print(f"\n--- LAYER 2: Matching Products ---")
    fig, axes = plt.subplots(1, n_products, figsize=(3.5 * n_products, 4))
    if n_products == 1: axes = [axes]

    for i, r in enumerate(product_results):
        pid = int(r['doc_id'])
        row = product_df.iloc[pid]
        print(f"  {i+1}. (score: {r['score']:.4f}) {row['productDisplayName']} | {row['baseColour']} {row['articleType']}")

        img = product_dataset[int(product_df.iloc[pid].name)]['image']
        axes[i].imshow(img)
        axes[i].set_title(f"{row['productDisplayName'][:28]}\n{row['baseColour']} | {row['articleType']}\nscore: {r['score']:.3f}",
                         fontsize=8, fontweight='bold')
        axes[i].axis('off')

    plt.suptitle(f"Products for: {query}", fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
stylefinder_visual("I have an hourglass figure and need something elegant for a summer wedding")

In [ ]:
stylefinder_visual("plus-size woman preparing for a job interview")

In [ ]:
stylefinder_visual("comfortable hiking outfit for cool autumn weather")

In [ ]:
stylefinder_visual("non-binary person minimalist outfit for an art gallery opening")

## 7. SQL + FAISS Hybrid Search Demo

In [ ]:
def filtered_visual(query, gender=None, article_type=None, colour=None, n=5):
    print(f"Query: {query}")
    filters = []
    if gender: filters.append(f"gender={gender}")
    if article_type: filters.append(f"type={article_type}")
    if colour: filters.append(f"colour={colour}")
    print(f"SQL Filters: {', '.join(filters) if filters else 'none'}")
    print("-" * 50)

    results = engine.filtered_search(query, gender=gender, article_type=article_type, colour=colour, n=n)
    if not results:
        print("  No matching products.")
        return

    print(f"  Found {len(results)} results after SQL filtering + FAISS ranking\n")

    fig, axes = plt.subplots(1, len(results), figsize=(3.5 * len(results), 4))
    if len(results) == 1: axes = [axes]

    for i, r in enumerate(results):
        pid = int(r['doc_id'])
        row = product_df.iloc[pid]
        img = product_dataset[int(product_df.iloc[pid].name)]['image']
        axes[i].imshow(img)
        axes[i].set_title(f"{r['name'][:28]}\n{r['colour']} | {r['type']}\nscore: {r['score']:.3f}",
                         fontsize=8, fontweight='bold')
        axes[i].axis('off')

    plt.suptitle(f"SQL+FAISS: {query} ({', '.join(filters)})", fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

filtered_visual("elegant evening outfit", gender="Women", article_type="Dresses", colour="Black")

In [ ]:
filtered_visual("casual weekend shirt", gender="Men", article_type="Shirts", colour="Blue")

In [ ]:
filtered_visual("sporty running shoes", gender="Women", article_type="Sports Shoes")

## 8. Evaluation Results

In [ ]:
# Load evaluation set
with open("evaluation_set.json", "r") as f:
    eval_set = json.load(f)

# Run evaluation
rows = []
for entry in eval_set:
    results = engine.faiss_search(entry["query"], layer=entry["layer"], n=5)
    retrieved = [r["doc_id"] for r in results]
    expected = entry["expected_doc_ids"]
    top_score = results[0]["score"] if results else 0

    if expected:
        hits = len([d for d in retrieved if d in expected])
        recall = hits / len(expected)
        qtype = "Realistic"
    else:
        hits = 0
        recall = None
        qtype = "Adversarial"

    rows.append({
        "ID": entry["query_id"],
        "Query": entry["query"][:45] + "...",
        "Layer": entry["layer"],
        "Type": qtype,
        "Expected": len(expected),
        "Hits@5": hits,
        "Recall": f"{recall:.2f}" if recall is not None else "N/A",
        "Top Score": f"{top_score:.4f}"
    })

df_eval = pd.DataFrame(rows)

def color_recall(val):
    if val == "N/A": return "background-color: #FFE0B2"
    v = float(val)
    if v >= 0.8: return "background-color: #C8E6C9"
    if v >= 0.4: return "background-color: #FFF9C4"
    return "background-color: #FFCDD2"

display(HTML("<h4>Evaluation Results: 15 Queries</h4>"))
display(df_eval.style.applymap(color_recall, subset=["Recall"]).hide(axis='index'))

# Summary stats
realistic = df_eval[df_eval["Type"] == "Realistic"]
with_hits = realistic[realistic["Hits@5"].astype(int) > 0]
print(f"\nRealistic queries with at least 1 hit: {len(with_hits)}/{len(realistic)}")
total_hits = realistic["Hits@5"].astype(int).sum()
total_expected = realistic["Expected"].astype(int).sum()
print(f"Overall recall: {total_hits}/{total_expected} = {total_hits/total_expected:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

realistic_rows = [r for r in rows if r["Type"] == "Realistic"]
queries = [r["ID"] + "\n" + r["Query"][:25] for r in realistic_rows]
recalls = [float(r["Recall"]) for r in realistic_rows]
colors = ['#43A047' if r >= 0.8 else '#FFC107' if r >= 0.4 else '#E53935' for r in recalls]

bars = ax.bar(queries, recalls, color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Recall@5", fontsize=12)
ax.set_title("Recall per Query (Realistic Only)", fontsize=14, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% threshold')

for bar, val in zip(bars, recalls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.2f}", ha='center', fontsize=9, fontweight='bold')

legend_patches = [
    mpatches.Patch(color='#43A047', label='High (>= 0.8)'),
    mpatches.Patch(color='#FFC107', label='Medium (>= 0.4)'),
    mpatches.Patch(color='#E53935', label='Low (< 0.4)')
]
ax.legend(handles=legend_patches, loc='upper right')
plt.xticks(fontsize=8, rotation=0)
plt.tight_layout()
plt.savefig("recall_per_query.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Interactive UMAP (Plotly)

In [ ]:
# Style embeddings with hover
labels = [f"{style_df.iloc[i]['input'][:50]}... | {style_df.iloc[i]['context']}" for i in range(len(style_df))]
groups = [occasion_group(style_df.iloc[i]['context']) for i in range(len(style_df))]

fig = px.scatter(x=style_2d[:, 0], y=style_2d[:, 1], color=groups, hover_name=labels,
                 title="Interactive UMAP: Style Recommendations (hover for details)",
                 color_discrete_map=occ_colors, opacity=0.6)
fig.update_traces(marker_size=5)
fig.update_layout(height=600)
fig.show()

In [ ]:
engine.close()
print("Done.")